<a href="https://colab.research.google.com/github/Lagnajit09/100x_AI_ML/blob/main/TrainingModel01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Building a simple model using the concepts of Tokenization, Positional Embeddings (Sin/Cos waves), Transformers (Self-Attention) and Feed forward network (ReLU).

# Step 1 — Building a Simple Tokenizer
A tokenizer takes a raw string of text and breaks it into smaller pieces (tokens). For example, Byte Pair Encoding (BPE) which breaks words into subwords — like "unhappiness" → "un", "happi", "ness".

In [1]:
# Step 1: A simple word-level tokenizer

text = "I love code and I love AI"

# Split into words (simplest possible tokenization)
tokens = text.lower().split()
print("Tokens:", tokens)

Tokens: ['i', 'love', 'code', 'and', 'i', 'love', 'ai']


# Step 2 — Building a Vocabulary
A neural network can't read the word "code". It only understands numbers. So we need a lookup table that assigns every unique word an integer ID.

In [2]:
# Step 2: Build a vocabulary (unique words → integer IDs)

# Get unique words, sorted for consistency
vocab = sorted(set(tokens))
print("Vocabulary:", vocab)

# Create two dictionaries — one for each direction
word_to_id = {word: idx for idx, word in enumerate(vocab)}
id_to_word = {idx: word for word, idx in word_to_id.items()}

print("\nWord → ID mapping:")
for word, idx in word_to_id.items():
    print(f"  '{word}' → {idx}")

Vocabulary: ['ai', 'and', 'code', 'i', 'love']

Word → ID mapping:
  'ai' → 0
  'and' → 1
  'code' → 2
  'i' → 3
  'love' → 4


In [3]:
# Convert tokens to IDs
token_ids = [word_to_id[token] for token in tokens]
print("Token IDs:", token_ids)

# Convert to a PyTorch tensor
import torch
token_ids_tensor = torch.tensor(token_ids)
print("As tensor:", token_ids_tensor)

Token IDs: [3, 4, 2, 1, 3, 4, 0]
As tensor: tensor([3, 4, 2, 1, 3, 4, 0])


# Step 3 — The Embedding Layer
Here's where numbers transform into meaning. Right now we have [3, 4, 2, 1, 3, 4, 0] — just flat integers. These don't capture any relationship between words. The number 3 isn't "closer" to 4 than to 0 in any meaningful way.

The embedding layer is essentially a big table with one row per word in our vocabulary, where each row is a vector of numbers

### What nn.Embedding(vocab_size, d_model) does:
It creates a matrix of random numbers with shape (vocab_size, d_model). For GPT with 50,000 tokens and 768 dimensions, it would create a (50,000, 768) table. That's it. Nothing fancy — just a big table of random numbers, one row per word.

```code
Think of it as a massive spreadsheet:
Row 0    (token "the"):  [0.23, 0.87, 0.11, ... 768 numbers]
Row 1    (token "of"):   [0.56, 0.32, 0.78, ... 768 numbers]
Row 2    (token "and"):  [0.41, 0.65, 0.29, ... 768 numbers]
...
Row 49999 (token "zzz"): [0.12, 0.44, 0.93, ... 768 numbers]
```

*Because nn.Embedding is part of nn.Module, PyTorch tracks gradients on that table. So during training, when backpropagation happens, it doesn't update all 50,000 rows. It only updates the rows that were actually looked up in that training step. If your sentence only used words 3, 4, 2, 1, and 0 — only those 5 rows get nudged. The other 49,995 rows stay untouched for that step.*


In [4]:
# Step 3: Embedding layer

vocab_size = len(vocab)     # 5 unique words
d_model = 4                 # each word gets a 4-dimensional vector

# nn.Embedding is just a fancy lookup table
embedding_layer = torch.nn.Embedding(vocab_size, d_model)

print("Embedding table (randomly initialized):")
print(embedding_layer.weight)
print("Shape:", embedding_layer.weight.shape)

Embedding table (randomly initialized):
Parameter containing:
tensor([[ 0.9512,  0.9046,  0.4908, -1.2417],
        [ 0.5879, -0.6154,  0.3838, -0.2604],
        [-0.2237, -0.2056, -0.1116, -3.0628],
        [-0.7281,  0.2815, -0.2298,  0.7422],
        [ 0.1942, -1.2535, -0.6809,  1.0254]], requires_grad=True)
Shape: torch.Size([5, 4])


### Now what happens when you call embedding_layer(token_ids_tensor)?

This is just a lookup operation. It does NOT create a new table. It goes to the existing table and picks out the rows you asked for.
Exactly what's happening:
```python
# Our token IDs:    [3, 4, 2, 1, 3, 4, 0]
#                    i love code and  i love ai

# What embedding_layer(token_ids_tensor) actually does:
# "Give me row 3" → grabs the vector for 'i'
# "Give me row 4" → grabs the vector for 'love'
# "Give me row 2" → grabs the vector for 'code'
# "Give me row 1" → grabs the vector for 'and'
# "Give me row 3" → grabs the vector for 'i'    (SAME row as before!)
# "Give me row 4" → grabs the vector for 'love' (SAME row as before!)
# "Give me row 0" → grabs the vector for 'ai'
```

The shape should be (7, 4) — 7 tokens in our sentence, each now represented by a 4-dimensional vector. Here's the critical thing: word "i" at position 0 and word "i" at position 4 have the exact same embedding. Because they're the same word, they look up the same row in the table.

And that's a problem. Think about this sentence: "The bank by the river" vs "I went to the bank." The word "bank" should mean different things based on position and context. Embeddings alone can't handle this — they just say "bank is bank."

Context comes from attention. But there's still the position problem — the model needs to know that the first "i" and the second "i" are in different positions.
That's what positional embeddings solve.

In [5]:
# Look up embeddings for our sentence
token_embeddings = embedding_layer(token_ids_tensor)

print("Input IDs:", token_ids_tensor)
print("\nToken embeddings:")
print(token_embeddings)
print("Shape:", token_embeddings.shape)

Input IDs: tensor([3, 4, 2, 1, 3, 4, 0])

Token embeddings:
tensor([[-0.7281,  0.2815, -0.2298,  0.7422],
        [ 0.1942, -1.2535, -0.6809,  1.0254],
        [-0.2237, -0.2056, -0.1116, -3.0628],
        [ 0.5879, -0.6154,  0.3838, -0.2604],
        [-0.7281,  0.2815, -0.2298,  0.7422],
        [ 0.1942, -1.2535, -0.6809,  1.0254],
        [ 0.9512,  0.9046,  0.4908, -1.2417]], grad_fn=<EmbeddingBackward0>)
Shape: torch.Size([7, 4])


# Step 4 — Positional Embeddings
We generate a unique "position signature" for each position using sine and cosine waves of different frequencies. Position 0 gets one pattern, position 1 gets a different pattern, position 6 gets yet another. These patterns are crafted so the model can easily figure out both absolute position ("I'm at position 3") and relative distance ("these two words are 2 positions apart").

In [6]:
# Step 4: Positional Encoding using sin/cos

import math

def create_positional_encoding(max_len, d_model):
    """
    Creates the sin/cos positional encoding table.
    max_len: longest sentence we'll handle
    d_model: embedding dimension (must match token embeddings)
    """
    pe = torch.zeros(max_len, d_model)

    for pos in range(max_len):          # for each position in the sentence
        for i in range(0, d_model, 2):  # for each pair of dimensions
            # The "wavelength" changes with dimension
            denominator = 10000 ** (i / d_model)

            pe[pos, i]     = math.sin(pos / denominator)  # even dimensions
            pe[pos, i + 1] = math.cos(pos / denominator)  # odd dimensions

    return pe

# Create positional encodings for up to 10 positions
pe = create_positional_encoding(max_len=10, d_model=4)

print("Positional encoding for first 7 positions:")
print(pe[:7])
print("Shape:", pe[:7].shape)

Positional encoding for first 7 positions:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998],
        [ 0.1411, -0.9900,  0.0300,  0.9996],
        [-0.7568, -0.6536,  0.0400,  0.9992],
        [-0.9589,  0.2837,  0.0500,  0.9988],
        [-0.2794,  0.9602,  0.0600,  0.9982]])
Shape: torch.Size([7, 4])


### Why adding the positional embeddings to the token embeddings instead of stacking them like (token_embeddings, position,embeddings)?

The deeper reason is about efficiency and information blending. When you add, the position information gets baked into the same 4 numbers that carry meaning. The model is forced to learn embeddings where meaning and position can coexist in the same space. This keeps the model smaller (no extra dimensions) and importantly, when attention later compares two words, it's comparing one vector that contains both "what" and "where" — not two separate pieces it has to figure out how to combine.

In [7]:
# Step 5: Combine token embeddings + positional encodings

# Take only the positions we need (7 tokens)
positional_encoding = pe[:len(tokens)]

# THE KEY OPERATION: add them together
final_embeddings = token_embeddings + positional_encoding

print("Token embeddings (what the word means):")
print(token_embeddings[0], "← first 'i'")
print(token_embeddings[4], "← second 'i'")

print("\nPositional encodings (where the word is):")
print(positional_encoding[0], "← position 0")
print(positional_encoding[4], "← position 4")

print("\nFinal embeddings (meaning + position):")
print(final_embeddings[0], "← 'i' at position 0")
print(final_embeddings[4], "← 'i' at position 4")

print("\nAre the two 'i' embeddings the same?",
      torch.equal(final_embeddings[0], final_embeddings[4]))

print("final_embeddings shape: ", final_embeddings.shape)

Token embeddings (what the word means):
tensor([-0.7281,  0.2815, -0.2298,  0.7422], grad_fn=<SelectBackward0>) ← first 'i'
tensor([-0.7281,  0.2815, -0.2298,  0.7422], grad_fn=<SelectBackward0>) ← second 'i'

Positional encodings (where the word is):
tensor([0., 1., 0., 1.]) ← position 0
tensor([-0.7568, -0.6536,  0.0400,  0.9992]) ← position 4

Final embeddings (meaning + position):
tensor([-0.7281,  1.2815, -0.2298,  1.7422], grad_fn=<SelectBackward0>) ← 'i' at position 0
tensor([-1.4849, -0.3721, -0.1898,  1.7414], grad_fn=<SelectBackward0>) ← 'i' at position 4

Are the two 'i' embeddings the same? False
final_embeddings shape:  torch.Size([7, 4])
